# Tetris OpenEnv - Training with Unsloth + GRPO

Train an LLM agent to play Tetris using GRPO (Group Relative Policy Optimization).

**Environment**: Tetris on HF Spaces via OpenEnv 0.2.1
**Model**: Qwen2.5-7B-Instruct (4-bit via Unsloth)
**Training**: GRPO via TRL

Runtime: GPU T4 (free tier Colab)

In [10]:
# Cell 1: Install dependencies (remove Unsloth, use standard HF stack)
!pip uninstall unsloth unsloth-zoo -y -q
!pip install peft trl openenv-core datasets accelerate -q


In [11]:
# Cell 2: Load model (standard HF stack — no Unsloth bug)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491


In [12]:
# Cell 3: Define Tetris client (standalone, no local package needed)
import json
import websockets.sync.client as ws_client

TETRIS_URL = "wss://vortexedsquirrel-tetris-env.hf.space/ws"

class TetrisClient:
    """Lightweight Tetris env client for Colab."""
    def __init__(self, url=TETRIS_URL):
        self.url = url
        self.ws = None

    def connect(self):
        self.ws = ws_client.connect(self.url, open_timeout=30)
        return self

    def _send_recv(self, msg):
        self.ws.send(json.dumps(msg))
        return json.loads(self.ws.recv(timeout=30))

    def reset(self, seed=None):
        data = {"seed": seed} if seed else {}
        resp = self._send_recv({"type": "reset", "data": data})
        d = resp["data"]
        obs = d.get("observation", d)
        return {"observation": obs, "reward": d.get("reward", 0), "done": d.get("done", False)}

    def step(self, action_str):
        resp = self._send_recv({"type": "step", "data": {"action": action_str, "metadata": {}}})
        d = resp["data"]
        obs = d.get("observation", d)
        return {"observation": obs, "reward": d.get("reward", 0), "done": d.get("done", False)}

    def close(self):
        if self.ws:
            self.ws.close()

# Quick test
client = TetrisClient().connect()
r = client.reset(seed=42)
print(f"Connected! Piece: {r['observation']['current_piece']}")
r = client.step("drop")
print(f"Step OK. Reward: {r['reward']}, Done: {r['done']}")
client.close()
print("Environment connection verified!")

Connected! Piece: L
Step OK. Reward: -7, Done: False
Environment connection verified!


In [13]:
# Cell 4: Generate training prompts — single-char actions, 60 fixed
import random

!wget -q -O game_engine.py https://raw.githubusercontent.com/OutOfMystic/tetris-openenv/main/src/tetris_env/server/game_engine.py

from game_engine import TetrisEnv

# Action encoding: 1 char = 1 action
ACTION_MAP = {'L': 'left', 'R': 'right', 'C': 'rotate_cw', 'W': 'rotate_ccw', 'D': 'drop', 'S': 'down'}
ACTION_CHARS = list(ACTION_MAP.keys())

SYSTEM_PROMPT = """You are a Tetris AI. Output exactly 60 actions as single letters separated by spaces.
L=left R=right C=rotate_cw W=rotate_ccw D=drop S=down
Example: L L C D R R D L D S D L C D R D ...
Strategy: fill complete rows. Multiple lines at once = bonus.
Output ONLY the 60 letters, nothing else."""

def make_prompt(result):
    return f"""Board:
{result['board']}

Piece: {result['current_piece']} Next: {result['next_piece']}
Score: {result['score']} Lines: {result['total_lines']} Height: {result['max_height']} Holes: {result['holes']}

Your 60 actions:"""

def generate_training_prompts(n_prompts=400):
    prompts = []
    for i in range(n_prompts):
        env = TetrisEnv(seed=i)
        result = env.reset(seed=i)
        rng = random.Random(i)
        moves = rng.randint(0, 4)
        direction = rng.choice(["left", "right"])
        for _ in range(moves):
            env.step(direction)
        result = env._make_result(0)
        prompts.append({
            "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                       {"role": "user", "content": make_prompt(result)}],
        })
    print(f"Generated {len(prompts)} prompts")
    return prompts

train_prompts = generate_training_prompts(400)


Generated 400 prompts


In [14]:
# Cell 5: Create HF Dataset from prompts
from datasets import Dataset

dataset = Dataset.from_list(train_prompts)
print(f"Dataset size: {len(dataset)}")
print(f"Example prompt (user message):")
print(dataset[0]['prompt'][1]['content'][:300])

Dataset size: 400
Example prompt (user message):
Board:
+----------+
|..........|
|..........|
|..........|
|........@.|
|........@.|
|.......@@.|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
|..........|
+----------+

Piece:


In [15]:
# Cell 6: Reward function — parse single chars, exactly 60 actions

NUM_ACTIONS = 60

def tetris_reward_func(completions, **kwargs):
    rewards = []
    for i, completion in enumerate(completions):
        text = completion[0]['content'].strip().upper() if isinstance(completion, list) else str(completion).strip().upper()

        # Parse single-char actions
        chars = [ch for ch in text.split() if ch in ACTION_MAP]
        # Also try without spaces (e.g. "LLCDRRД")
        if len(chars) < 5:
            chars = [ch for ch in text if ch in ACTION_MAP]

        # Pad to 60 or truncate to 60
        if len(chars) < NUM_ACTIONS:
            chars += ['D'] * (NUM_ACTIONS - len(chars))  # pad with drop
        chars = chars[:NUM_ACTIONS]

        # Play on engine
        env = TetrisEnv(seed=i)
        env.reset(seed=i)

        total_lines = 0
        steps_played = 0
        game_over = False

        for ch in chars:
            result = env.step(ACTION_MAP[ch])
            steps_played += 1
            total_lines = result['total_lines']
            if result['done']:
                game_over = True
                break

        reward = 0.0
        reward += total_lines * 200.0
        reward += steps_played * 0.5
        reward -= result['holes'] * 2.0
        reward -= result['max_height'] * 1.0
        if game_over:
            reward -= 50.0

        rewards.append(reward)

    return rewards

# Test
test = [[{"content": "L L C D R R D L D S D " * 6}],
        [{"content": "D D D D D D D D D D " * 6}],
        [{"content": "garbage text no actions"}]]
print(f"Test rewards: {tetris_reward_func(test)}")


Test rewards: [-157.5, -138.5, -94.0]


In [ ]:
# Cell 7: Configure and run GRPO training (A100, with reward logging)
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback

class RewardLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = state.global_step
        parts = []
        for k in sorted(logs):
            if k != 'loss' and k != 'learning_rate' and k != 'epoch':
                parts.append(f"{k}={logs[k]:.3f}")
        if parts:
            print(f"[Step {step}] {', '.join(parts)}")

training_args = GRPOConfig(
    output_dir="tetris-agent-grpo",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_generations=8,
    max_completion_length=128,
    learning_rate=5e-6,
    logging_steps=10,             # log every 10 steps (less spam)
    save_steps=50,
    bf16=True,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[tetris_reward_func],
    args=training_args,
    train_dataset=dataset,
    callbacks=[RewardLogger()],
)

print("Starting GRPO training...")
trainer.train()
print("Training complete!")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting GRPO training...


Step,Training Loss
10,-0.010562
20,0.027683
30,-0.002452
40,0.018359
50,-0.016683
60,-0.039459
70,-0.054543
80,-0.001451
90,-0.050575
100,-0.004233


[Step 10] clip_ratio/high_max=0.000, clip_ratio/high_mean=0.000, clip_ratio/low_mean=0.000, clip_ratio/low_min=0.000, clip_ratio/region_mean=0.000, completions/clipped_ratio=0.338, completions/max_length=128.000, completions/max_terminated_length=101.100, completions/mean_length=91.037, completions/mean_terminated_length=72.645, completions/min_length=49.600, completions/min_terminated_length=49.600, entropy=0.123, frac_reward_zero_std=0.000, grad_norm=0.269, num_tokens=23451.000, reward=-77.094, reward_std=64.314, rewards/tetris_reward_func/mean=-77.094, rewards/tetris_reward_func/std=64.314, step_time=10.472
[Step 20] clip_ratio/high_max=0.000, clip_ratio/high_mean=0.000, clip_ratio/low_mean=0.000, clip_ratio/low_min=0.000, clip_ratio/region_mean=0.000, completions/clipped_ratio=0.375, completions/max_length=128.000, completions/max_terminated_length=89.000, completions/mean_length=91.975, completions/mean_terminated_length=70.433, completions/min_length=55.100, completions/min_termi

Step,Training Loss
10,-0.010562
20,0.027683
30,-0.002452
40,0.018359
50,-0.016683
60,-0.039459
70,-0.054543
80,-0.001451
90,-0.050575
100,-0.004233


In [1]:
# Cell 8: Plot reward curve
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_logs = [l for l in logs if 'loss' in l]
reward_logs = [l for l in logs if 'reward' in l or 'rewards/mean' in l]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if train_logs:
    axes[0].plot([l.get('step', i) for i, l in enumerate(train_logs)],
                 [l['loss'] for l in train_logs])
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')

reward_key = 'reward' if reward_logs and 'reward' in reward_logs[0] else 'rewards/mean'
if reward_logs:
    axes[1].plot([l.get('step', i) for i, l in enumerate(reward_logs)],
                 [l.get(reward_key, 0) for l in reward_logs])
    axes[1].set_title('Mean Reward (should go up!)')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Reward')

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150)
plt.show()
print("Reward curve saved to reward_curve.png")

NameError: name 'trainer' is not defined

In [ ]:
# Cell 9: Test trained agent
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

client = TetrisClient().connect()
result = client.reset(seed=123)

total_reward = 0
for step_num in range(50):
    if result['done']:
        break

    prompt_text = make_prompt(result['observation'])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text}
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    outputs = model.generate(inputs, max_new_tokens=16, temperature=0.1)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip().lower()

    # Parse action
    action = "noop"
    for valid in VALID_ACTIONS:
        if valid in response:
            action = valid
            break

    result = client.step(action)
    total_reward += result['reward']

    if step_num % 10 == 0:
        print(f"Step {step_num}: action={action}, reward={result['reward']:.1f}, score={result['observation']['score']}")

print(f"\nGame over! Total reward: {total_reward:.1f}, Score: {result['observation']['score']}, Lines: {result['observation']['total_lines']}")
print(f"\nFinal board:")
print(result['observation']['board'])
client.close()

In [ ]:
# Cell 10: Push trained model to HF Hub
model.push_to_hub("VortexedSquirrel/tetris-agent-grpo")
tokenizer.push_to_hub("VortexedSquirrel/tetris-agent-grpo")
print("Model pushed to https://huggingface.co/VortexedSquirrel/tetris-agent-grpo")